# Visualization of Interacting Argon Atoms
> Harrison B. Prosper<br>
> March 2026

## Introduction
In this notebook we visualize the motion of argon atoms [1] in a microscopic spherical container. See notebook `argon_simulator.ipynb` for details.


## Coordinate system
For this project, please choose the coordinate system so that $+z$ points upwards and the $x-y$ plane is horizontal. 

## References
 1. A. Rahman, *Correlations in the Motion of Atoms in Liquid Argon*, Phys. Rev. 136, A405, 1964.

## Tips

  * Use __esc r__ to disable a cell
  * Use __esc y__ to reactivate it
  * Use __esc m__ to go to markdown mode
  * Shift + return to execute a cell

In [1]:
import vpython as vp
import numpy as np
import h5py

from comphyslab.graphics import Scene, Sim, Controls, Zoom, Histogram, Graph, J, K

from comphyslab.newton import min_separation, \
maxwell_distribution, radial_distribution

from comphyslab.vectors import magnitude
from comphyslab.utils import Bag, SimReader

/Users/harry/miniconda3/envs/comphys/lib/python3.13/site-packages/vpython/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


<IPython.core.display.Javascript object>

##  Live plots

In [2]:
def setup_live_plots(bg):
    # For histogram and graphs
    bg.draw_rate = 6   
    bg.bufsize= 200
    
    # For velocity distribution
    bg.vbins = 40
    bg.vmin  = 0.0
    bg.vmax  = 8.0
    bg.vymin = 0.0
    bg.vymax = 0.20

    # For radial distribution
    bg.rmin  = 0.0
    bg.rmax  = float(np.ceil(bg.R))
    bg.rbins = int(10 * bg.rmax)
    bg.rymin = 0.0
    bg.rymax = 3.0

    # Speeds 
    vmag = magnitude(bg.v)
    T    = np.mean(vmag**2) / 3  # dimensionless temperature
    y, _ = maxwell_distribution(T, bg.vmin, bg.vmax, bg.vbins)
    bg.T = bg.T2K * T

    H = 200
    W = int(H*16/9)

    gfx = bg.gfx
    
    gfx.hspeed = Histogram(
        title=f'\nSpeed (T: {bg.T:8.2f} K)', 
        xtitle=f'Speed (vc)', 
        ytitle='f(v)',
        nbins=bg.vbins, 
        xmin=bg.vmin, 
        xmax=bg.vmax, 
        ymin=bg.vymin,
        ymax=bg.vymax,
        bufsize=bg.bufsize,
        height=H,
        width=W,
        align='left',
        density=True)

    bg.gfx.hspeed.fill(vmag)
    bg.gfx.hspeed.draw()

    # Maxwell distribution
    bg.gfx.theory = Graph(
        title='Speed distribution', 
        xtitle='Speed (vc)', 
        ytitle='Density', 
        npoints=bg.vbins, 
        xmin=bg.vmin, 
        xmax=bg.vmax, 
        graph=gfx.hspeed.g)

    bg.gfx.theory.fill(y)
    bg.gfx.theory.draw() 
    
    # Radial distribution
    gfx.gradial = Graph(
        title='Radial distribution', 
        xtitle=f'r (units: {1e9*bg.sigma:6.2f} nm)', 
        ytitle='g(r)', 
        npoints=bg.rbins, 
        xmin=bg.rmin, 
        xmax=bg.rmax,
        ymin=bg.rymin, 
        ymax=bg.rymax,
        color=vp.color.blue,
        height=H,
        width=W,
        align='left',
        bufsize=bg.bufsize)

    y, _ = radial_distribution(
        bg.rho, bg.r, rmax=bg.rmax, nbins=bg.rbins, rcore=bg.Rcore)
    gfx.gradial.fill(y)
    gfx.gradial.draw()

def update_live_plots(bg):
    if bg.frame % bg.draw_rate != 0:
        return
    gfx = bg.gfx
    
    # Fill histograms
    vmag = magnitude(bg.v)
    gfx.hspeed.fill(vmag)

    y, _ = radial_distribution(
        bg.rho, bg.r, rmax=bg.rmax, nbins=bg.rbins, rcore=bg.Rcore)
    gfx.gradial.fill(y)

    T = np.mean(vmag**2) / 3  # dimensionless temperature
    y, _ = maxwell_distribution(T, bg.vmin, bg.vmax, bg.vbins)
    gfx.theory.fill(y)

    # Draw histograms / graphs
    gfx.hspeed.draw()
    gfx.gradial.draw()
    gfx.theory.draw()

## Define the scene

In [3]:
def build_scene(bg):
    # ----------------------------------------------------
    # Required attributes
    # ----------------------------------------------------
    bg.rate   = 30         # No faster than "rate" frames/second
    bg.frame  = 0          # Frame counter
    bg.active = True       # Controled by Stop button 
    bg.update = False      # Controled by Start/Pause button

    # ----------------------------------------------------
    # Create empty scene
    # ----------------------------------------------------
    # All scene widgets must be placed in bag.gfx so that they
    # can be properly deleted when the animation is stopped.
    gfx = bg.gfx           # give bag.gfx a shorter name

    bg.height = 200
    bg.size   = bg.R       # Scale of scene (units of sigma)
    gfx.scene = Scene(
        'Argon Atoms\n', bg.size, up=K, height=int(1.2*bg.height))
    gfx.scene.forward = vp.vector(-1,0,0)
    
    # ----------------------------------------------------
    # Add scene elements
    # ----------------------------------------------------
    gfx.container = vp.sphere(
        pos=vp.vector(0.0,0.0,0.0), 
        radius=bg.R, color=vp.color.cyan, opacity=0.05)

    # Core of container
    bg.Rcore = bg.R * (1/3)**(1/3) 
    gfx.container_core = vp.sphere(
        pos=vp.vector(0.0,0.0,0.0), 
        radius=bg.Rcore, color=vp.color.magenta, opacity=0.05)
 
    # Model each atom as a small sphere
    gfx.atoms = []
    atom_radius = 0.02 * bg.R
    
    for i in range(bg.N):
        gfx.atoms.append(vp.sphere(
            pos=vp.vector(*bg.r[i]), 
            color=vp.color.yellow,
            radius=atom_radius))

    # Add control buttons (Stop, Start/Pause)
    controls = Controls(bg)

    gfx.b_stop  = vp.button(
        text="Stop",
        background=vp.color.red,
        pos=gfx.scene.title_anchor,
        bind=controls.stop)

    gfx.b_start_pause = vp.button(
        text="Start",
        background=vp.color.green,
        pos=gfx.scene.title_anchor,
        bind=controls.start_pause)

    # Add zoom slider
    gfx.zoom = Zoom(gfx.scene)
    
    setup_live_plots(bg)

## Scene update function

The $\texttt{update\_scene}$ function is responsible for updating all widgets in a scene and the $\texttt{rate}(n)$ function controls the frequency of updates. Updates will occur no faster than $n$ / second.

The standard update loop has the form:

```python
    while active:
        
        if update:
            
            update_scene(bg) 
            
        vp.rate(bg.rate) # allow updates no faster than bg.rate = 30 Hz
```

In [4]:
def update(bg):
    gfx = bg.gfx
    frame = bg.frame

    # Get data for current frame
    try:
        data =  bg.read(frame)
        t, bg.r[:], bg.v[:] = data.t, data.r, data.v
    except:
        bg.update = False
        return
        
    # Loop over graphical objects and update their positions.
    for i, atom in enumerate(gfx.atoms):
        x, y, z = bg.r[i]
        atom.pos.x = x
        atom.pos.y = y
        atom.pos.z = z

    # Update plots
    update_live_plots(bg)    

## RUN

In [5]:
def read_sim_data(filename):

    reader = SimReader(filename)
    print(reader)
    
    bg = reader.header()
    
    print(f'density:         {bg.rho*bg.mass/bg.sigma**3:10.3e} kg/m^3')
    print(f'number density:  {bg.rho:10.3e}/sigma^3')
    print(f'number of atoms: {bg.N:5d} atoms, R = {bg.R:6.3f} sigma')
    print(f'min(separation): {bg.rmin_sep:6.3f} sigma\n')

    T_reduced = float(bg.T / bg.T2K)
    print(f'Vrms:  {bg.vc*bg.vrms:5.1f} m/s,\tT: {bg.T:8.2f} K')
    print(f'Vrms:  {bg.vrms:5.1f} vc,\tT: {T_reduced:8.2f}')

    # Initial positions and velocities
    data = bg.read(0)
    bg.r = data.r
    bg.v = data.v
    return bg

filename = 'argon_gas_T0145_rho0050.h5'

bag = read_sim_data(filename)


Filename: argon_gas_T0145_rho0050.h5
  Attributes:
    N               : numpy.int32
    R               : numpy.float32
    T               : numpy.float32
    T2K             : numpy.float32
    dt              : numpy.float32
    epsilon         : numpy.float32
    equi_sep        : numpy.float32
    mass            : numpy.float32
    rho             : numpy.float32
    rmin_sep        : numpy.float32
    save_every      : numpy.int32
    sigma           : numpy.float32
    tc              : numpy.float32
    vc              : numpy.float32
    vrms            : numpy.float32

  Datasets:
    U               : h5py._hl.dataset.Dataset
    impulse         : h5py._hl.dataset.Dataset
    r               : h5py._hl.dataset.Dataset
    v               : h5py._hl.dataset.Dataset

density:          5.000e+01 kg/m^3
number density:   2.938e-02/sigma^3
number of atoms:   297 atoms, R = 13.414 sigma
min(separation):  3.250 sigma

Vrms:  299.6 m/s,	T:   145.00 K
Vrms:    1.9 vc,	T:     1.21


In [6]:
%%timeit
b = bag.read(0)

23.8 μs ± 1.29 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [7]:
build_scene(bag)

sim = Sim(bag, update)

sim.run()

<IPython.core.display.Javascript object>

Animation ended!
